# CHƯƠNG 4: ĐÁNH GIÁ CHUYÊN SÂU, PHÂN TÍCH LỖI VÀ ĐỐI SÁNH KẾT QUẢ
**Người thực hiện:** Hoàng Xuân Phi Long (Nhóm trưởng — Nhóm 01)  
**Học phần:** Học máy (Machine Learning) — Trường Đại học Công Thương TP. Hồ Chí Minh (HUIT)

---
Notebook này thực hiện kiểm định độc lập toàn bộ các mô hình đã huấn luyện và tinh chỉnh ở Chương 3 trên tập kiểm thử tách biệt (`X_test_raw.csv`, `y_test.csv`, $n = 120$ mẫu — tỷ lệ 70/30), bao gồm:
1. **Mục 4.1:** Đánh giá toàn diện 9 mô hình (3 mô hình Tuned và 6 mô hình Baseline) trên tập Test độc lập ($n = 120$: 75 CKD, 45 Not CKD) kèm **Khoảng tin cậy 95% Bootstrap ($B=1.000$)** và kiểm định độ ổn định qua **25-Fold Repeated Stratified Cross-Validation** trên tập Train ($n = 280$).
2. **Mục 4.2:** Phân tích Ma trận nhầm lẫn (Confusion Matrix), đánh giá chi tiết các ca Âm tính giả (FN) và Dương tính giả (FP).
3. **Mục 4.3:** Phân tích đường cong ROC-AUC, Precision-Recall (PR-AUC) và khảo sát bài toán tinh chỉnh ngưỡng quyết định xác suất ($\tau \in [0.10, 0.90]$).
4. **Mục 4.4:** Phân tích lỗi chuyên sâu ở cấp độ từng hồ sơ bệnh nhân qua `split_manifest.csv` (truy vết ca bệnh trẻ tuổi giai đoạn sớm `ID = 102`, ca bệnh khuyết huyết học `ID = 224` và cơ chế đồng hóa về trung vị trên `ID = 324`).
5. **Mục 4.5:** Giải thích mô hình (Feature Importance theo XGBoost Gain, Random Forest Gini và Permutation Importance) và thực nghiệm định lượng sai lệch do rò rỉ dữ liệu (Data Leakage) thường gặp trên Kaggle.

In [1]:
import os
import sys
import warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

warnings.filterwarnings('ignore')

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..')) if os.path.basename(os.getcwd()) == 'notebook' else os.path.abspath('.')
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import ckd_preprocessing
from src.evaluate_chapter4 import (
    setup_tufte_style, load_all_splits, build_and_fit_models,
    evaluate_comprehensive_test_set, evaluate_repeated_cv_stability,
    analyze_thresholds, perform_error_analysis, analyze_feature_importance,
    run_kaggle_leakage_audit
)

setup_tufte_style()
X_train_raw, y_train, X_test_raw, y_test, manifest = load_all_splits()
print(f"Train shape: {X_train_raw.shape} (CKD={int(y_train.sum())}, NotCKD={int((y_train==0).sum())})")
print(f"Test shape : {X_test_raw.shape} (CKD={int(y_test.sum())}, NotCKD={int((y_test==0).sum())})")

Train shape: (280, 24) (CKD=175, NotCKD=105)
Test shape : (120, 24) (CKD=75, NotCKD=45)


## 4.1. Kết quả đánh giá toàn diện trên tập Kiểm thử độc lập ($n = 120$) và Độ ổn định 25-Fold Repeated CV
Ta nạp kết quả kiểm định của cả 3 mô hình đã tinh chỉnh (`XGBoost (Tuned)`, `Random Forest (Tuned)`, `SVM (Tuned)`) cùng 6 mô hình cơ sở (`Baseline`) trên tập kiểm thử độc lập `X_test_raw.csv` ($n = 120$) kèm khoảng tin cậy 95% Bootstrap ($B = 1.000$ lần lấy mẫu lại) và kết quả kiểm chứng chéo lặp 5×5 (25 folds) trên tập Train ($n = 280$).

In [2]:
df_test = pd.read_csv(os.path.join(PROJECT_ROOT, 'reports', 'tables', 'ch4_test_comprehensive_metrics.csv'))
df_stab = pd.read_csv(os.path.join(PROJECT_ROOT, 'reports', 'tables', 'ch4_repeated_cv_stability.csv'))

print("=== BẢNG 4.1: KẾT QUẢ ĐÁNH GIÁ TOÀN DIỆN TRÊN TẬP TEST ĐỘC LẬP (n = 120) ===")
cols_show = ['Model', 'TP', 'TN', 'FP', 'FN', 'Accuracy', 'Accuracy_95CI', 'Recall', 'Recall_95CI', 'Specificity', 'F1_Score', 'F2_Score', 'ROC_AUC', 'PR_AUC', 'Brier_Score']
display(df_test[cols_show])

print("\n=== BẢNG 4.2: KIỂM ĐỊNH ĐỘ ỔN ĐỊNH QUA 25-FOLD REPEATED STRATIFIED CV (n = 280) ===")
display(df_stab)

=== BẢNG 4.1: KẾT QUẢ ĐÁNH GIÁ TOÀN DIỆN TRÊN TẬP TEST ĐỘC LẬP (n = 120) ===


,Model,TP,TN,FP,FN,Accuracy,Accuracy_95CI,Recall,Recall_95CI,Specificity,F1_Score,F2_Score,ROC_AUC,PR_AUC,Brier_Score
0,XGBoost (Tuned),74,45,0,1,0.991667,"[0.9750, 1.0000]",0.986667,"[0.9558, 1.0000]",1.000000,0.993289,0.989305,0.999704,0.999825,0.008773
1,Random Forest (Tuned),74,44,1,1,0.983333,"[0.9583, 1.0000]",0.986667,"[0.9558, 1.0000]",0.977778,0.986667,0.986667,0.999704,0.999825,0.010747
2,SVM (Tuned),73,45,0,2,0.983333,"[0.9583, 1.0000]",0.973333,"[0.9315, 1.0000]",1.000000,0.986486,0.978552,0.999407,0.999654,0.014801
3,Logistic Regression (Baseline),74,45,0,1,0.991667,"[0.9750, 1.0000]",0.986667,"[0.9558, 1.0000]",1.000000,0.993289,0.989305,0.998519,0.999167,0.014804
4,K-Nearest Neighbors (Baseline),67,45,0,8,0.933333,"[0.8833, 0.9750]",0.893333,"[0.8133, 0.9571]",1.000000,0.943662,0.912807,0.999407,0.999298,0.039667
5,SVM (Baseline),73,45,0,2,0.983333,"[0.9583, 1.0000]",0.973333,"[0.9315, 1.0000]",1.000000,0.986486,0.978552,1.000000,1.000000,0.007027
6,Decision Tree (Baseline),73,45,0,2,0.983333,"[0.9583, 1.0000]",0.973333,"[0.9324, 1.0000]",1.000000,0.986486,0.978552,0.986667,0.990000,0.016667
7,Random Forest (Baseline),74,44,1,1,0.983333,"[0.9583, 1.0000]",0.986667,"[0.9558, 1.0000]",0.977778,0.986667,0.986667,0.999704,0.999825,0.010995
8,XGBoost (Baseline),74,45,0,1,0.991667,"[0.9750, 1.0000]",0.986667,"[0.9558, 1.0000]",1.000000,0.993289,0.989305,1.000000,1.000000,0.005788



=== BẢNG 4.2: KIỂM ĐỊNH ĐỘ ỔN ĐỊNH QUA 25-FOLD REPEATED STRATIFIED CV (n = 280) ===


,Model,Folds,Recall_Mean,Recall_Std,Recall_Min,ROC_AUC_Mean,ROC_AUC_Std,F1_Mean,F1_Std,Accuracy_Mean,Accuracy_Std
0,XGBoost (Tuned),25,0.970286,0.027381,0.914286,0.996952,0.004375,0.976382,0.017121,0.970714,0.021357
1,Random Forest (Tuned),25,0.985143,0.018286,0.942857,0.998857,0.002166,0.986296,0.014174,0.982857,0.017843
2,SVM (Tuned),25,0.992000,0.012829,0.971429,0.999619,0.000722,0.993141,0.009097,0.991429,0.011429
3,Logistic Regression (Baseline),25,0.989714,0.013714,0.971429,0.999565,0.000836,0.990840,0.010607,0.988571,0.013248
4,Decision Tree (Baseline),25,0.962286,0.035886,0.885714,0.966857,0.025749,0.972073,0.021555,0.965714,0.026206
5,K-Nearest Neighbors (Baseline),25,0.902857,0.041991,0.800000,0.988653,0.010126,0.948433,0.023371,0.939286,0.026245


## 4.2. Phân tích Ma trận nhầm lẫn (Confusion Matrix Analysis)
Tại ngưỡng mặc định $\tau = 0.50$ trên 120 mẫu kiểm thử độc lập (75 CKD, 45 Not CKD):
- **`XGBoost (Tuned)`** đạt $\text{TP} = 74/75$, $\text{TN} = 45/45$, $\text{FN} = 1$, $\text{FP} = 0$ ($\text{Accuracy} = 99.17\%$, $\text{Recall} = 98.67\%$, $\text{Specificity} = 100.0\%$, $\text{Precision} = 100.0\%$).
- **`Random Forest (Tuned)`** đạt $\text{TP} = 74/75$ ($\text{FN} = 1$, $\text{Recall} = 98.67\%$), nhưng phát sinh 1 ca Dương tính giả ($\text{TN} = 44/45$, $\text{FP} = 1$, $\text{Specificity} = 97.78\%$, $\text{Accuracy} = 98.33\%$).
- **`SVM (Tuned)`** đạt $\text{TN} = 45/45$ ($\text{FP} = 0$, $\text{Specificity} = 100.0\%$), nhưng bỏ sót 2 ca bệnh ($\text{TP} = 73/75$, $\text{FN} = 2$, $\text{Recall} = 97.33\%$, $\text{Accuracy} = 98.33\%$).

In [3]:
from PIL import Image
img_cm = Image.open(os.path.join(PROJECT_ROOT, 'reports', 'figures', 'ch4_confusion_matrices_tuned.png'))
plt.figure(figsize=(14, 4.5))
plt.imshow(img_cm)
plt.axis('off')
plt.tight_layout()
plt.show()

## 4.3. Phân tích đường cong ROC, Precision-Recall và Tinh chỉnh ngưỡng quyết định ($\tau$)
`XGBoost (Tuned)` và `Random Forest (Tuned)` cùng đạt diện tích dưới đường cong $\text{ROC-AUC} = 0.9997$ và $\text{PR-AUC} = 0.9998$ (trong khi `SVM (Tuned)` đạt $\text{ROC-AUC} = 0.9994$ và $\text{PR-AUC} = 0.9997$). Khảo sát biến thiên ngưỡng $\tau \in [0.10, 0.90]$ chứng minh:
- `XGBoost (Tuned)` duy trì ổn định $\text{TP} = 74/75$ ($98.67\%$) và $\text{TN} = 45/45$ ($100.0\%$) trên toàn bộ dải ngưỡng chuẩn $\tau \in [0.40, 0.60]$.
- Khi chuyển sang chế độ tầm soát cộng đồng ($\tau = 0.25$ cho `XGBoost` hoặc $\tau = 0.35$ cho `Random Forest`), hệ thống thu hồi được cả ca bệnh `ID = 102` để đạt tuyệt đối $\text{Recall} = 100.0\%$ ($75/75$ ca CKD, $\text{FN} = 0$).

In [4]:
df_thresh = pd.read_csv(os.path.join(PROJECT_ROOT, 'reports', 'tables', 'ch4_threshold_tuning.csv'))
display(df_thresh[df_thresh['Threshold'].isin([0.25, 0.35, 0.40, 0.45, 0.50, 0.60, 0.70])])

for fig_name in ['ch4_roc_pr_curves_comparison.png', 'ch4_threshold_tuning_curves.png']:
    img = Image.open(os.path.join(PROJECT_ROOT, 'reports', 'figures', fig_name))
    plt.figure(figsize=(13.5, 5.0))
    plt.imshow(img)
    plt.axis('off')
    plt.tight_layout()
    plt.show()

,Model,Threshold,TP,TN,FP,FN,Recall,Specificity,Precision,F1_Score,F2_Score,Youden_J
3,XGBoost (Tuned),0.25,75,43,2,0,1.000000,0.955556,0.974026,0.986842,0.994695,0.955556
5,XGBoost (Tuned),0.35,74,44,1,1,0.986667,0.977778,0.986667,0.986667,0.986667,0.964444
6,XGBoost (Tuned),0.40,74,45,0,1,0.986667,1.000000,1.000000,0.993289,0.989305,0.986667
7,XGBoost (Tuned),0.45,74,45,0,1,0.986667,1.000000,1.000000,0.993289,0.989305,0.986667
8,XGBoost (Tuned),0.50,74,45,0,1,0.986667,1.000000,1.000000,0.993289,0.989305,0.986667
9,XGBoost (Tuned),0.60,74,45,0,1,0.986667,1.000000,1.000000,0.993289,0.989305,0.986667
10,XGBoost (Tuned),0.70,73,45,0,2,0.973333,1.000000,1.000000,0.986486,0.978552,0.973333
16,Random Forest (Tuned),0.25,75,43,2,0,1.000000,0.955556,0.974026,0.986842,0.994695,0.955556
18,Random Forest (Tuned),0.35,75,44,1,0,1.000000,0.977778,0.986842,0.993377,0.997340,0.977778
19,Random Forest (Tuned),0.40,75,44,1,0,1.000000,0.977778,0.986842,0.993377,0.997340,0.977778


## 4.4. Phân tích lỗi mô hình chuyên sâu ở cấp độ bệnh nhân (Patient-level Error Analysis)
Truy vết qua `split_manifest.csv` cho thấy:
1. Ca bệnh duy nhất trên tập Test làm thất bại `XGBoost (Tuned)` và `Random Forest (Tuned)` ở ngưỡng chuẩn là bệnh nhân **`Original_ID = 102` (`Test_Position = 52`, 17 tuổi, mắc CKD giai đoạn sớm chưa có thiếu máu với `hemo = 13.9 g/dL`, `pcv = 52%`, `al = 0`, chỉ có `sc = 2.1 mg/dL` và `sg = 1.010`)**.
2. Ca bệnh **`Original_ID = 224` (`Test_Position = 110`, 34 tuổi, `sc = 2.2 mg/dL`, thiếu 5 cột gồm cả 4 chỉ số huyết học)** bị `SVM (Tuned)` và `KNN` bỏ sót ($\hat{p}_{\text{SVM}} = 0.4595$), nhưng được `XGBoost (Tuned)` phát hiện chính xác ($\hat{p}_{\text{XGB}} = 0.9659$).
3. Ca khỏe mạnh **`Original_ID = 324` (`Test_Position = 53`, 40 tuổi, thiếu 4 chỉ số huyết học `hemo, pcv, wc, rc`)** sau khi điền trung vị tập Train (`hemo = 12.6, pcv = 40.0`) đã khiến `Random Forest (Tuned)` báo động giả ($\hat{p}_{\text{RF}} = 0.5600 > 0.50$), trong khi `XGBoost (Tuned)` kiểm soát tốt ở mức $\hat{p}_{\text{XGB}} = 0.3936 < 0.50$ ($\text{FP} = 0$).

In [5]:
df_errors = pd.read_csv(os.path.join(PROJECT_ROOT, 'reports', 'tables', 'ch4_error_analysis_cases.csv'))
print("=== DANH SÁCH CÁC BẢN GHI BỊ DỰ ĐOÁN SAI HOẶC SÁT RANH GIỚI QUYẾT ĐỊNH ===")
display(df_errors[df_errors['Num_Models_Failed'] > 0])

img_err = Image.open(os.path.join(PROJECT_ROOT, 'reports', 'figures', 'ch4_error_cases_clinical_profile.png'))
plt.figure(figsize=(11.5, 7.0))
plt.imshow(img_err)
plt.axis('off')
plt.tight_layout()
plt.show()

=== DANH SÁCH CÁC BẢN GHI BỊ DỰ ĐOÁN SAI HOẶC SÁT RANH GIỚI QUYẾT ĐỊNH ===


,Split,Test_Position,Original_ID,Source_Row,True_Label,True_Class,Missing_Count_24,Missing_Percent,Missing_Features,Prob_XGBoost_Tuned,...,al,su,sc,bu,bgr,hemo,pcv,rc,htn,dm
0,Test,52,102,102,1,CKD (1),2,8.3,"rbc, rc",0.2946,...,0.0,0.0,2.1,32.0,92.0,13.9,52.0,NaN,no,no
1,Test,110,224,224,1,CKD (1),5,20.8,"rbc, hemo, pcv, wc, rc",0.9659,...,0.0,0.0,2.2,28.0,117.0,NaN,NaN,NaN,no,no
2,Test,53,324,324,0,Not CKD (0),4,16.7,"hemo, pcv, wc, rc",0.3936,...,0.0,0.0,0.7,15.0,119.0,NaN,NaN,NaN,no,no
3,Test,34,186,186,1,CKD (1),5,20.8,"bgr, hemo, pcv, wc, rc",0.6556,...,4.0,0.0,1.0,46.0,NaN,NaN,NaN,NaN,no,no
4,Test,8,65,65,1,CKD (1),6,25.0,"rbc, bgr, sod, pot, wc, rc",0.7704,...,1.0,0.0,1.1,20.0,NaN,15.0,48.0,NaN,no,no
5,Test,68,164,164,1,CKD (1),3,12.5,"bp, rbc, pc",0.8012,...,0.0,0.0,0.8,15.0,192.0,14.3,40.0,5.4,no,yes
6,Test,20,1,1,1,CKD (1),5,20.8,"rbc, bgr, sod, pot, rc",0.9267,...,4.0,0.0,0.8,18.0,NaN,11.3,38.0,NaN,no,no
7,Test,112,91,91,1,CKD (1),0,0.0,NaN,0.9622,...,4.0,1.0,1.7,26.0,210.0,16.1,52.0,5.6,no,no
8,Test,117,133,133,1,CKD (1),0,0.0,NaN,0.9853,...,4.0,0.0,5.3,125.0,118.0,12.0,37.0,8.0,yes,no
41,Train_OOF,84,319,319,0,Not CKD (0),4,16.7,"hemo, pcv, wc, rc",0.8437,...,0.0,0.0,1.1,15.0,138.0,NaN,NaN,NaN,no,no


## 4.5. Giải thích mô hình (Feature Importance) và Thực nghiệm định lượng Data Leakage trên Kaggle
### 4.5.1. Xếp hạng tầm quan trọng của 24 đặc trưng lâm sàng

In [6]:
df_feat24 = pd.read_csv(os.path.join(PROJECT_ROOT, 'reports', 'tables', 'ch4_feature_importance_24_grouped.csv'))
display(df_feat24.head(15))

img_fi = Image.open(os.path.join(PROJECT_ROOT, 'reports', 'figures', 'ch4_feature_importance_comparison.png'))
plt.figure(figsize=(13, 5.8))
plt.imshow(img_fi)
plt.axis('off')
plt.tight_layout()
plt.show()

,Original_Feature,XGBoost_Gain_Importance,RandomForest_Gini_Importance,Perm_Test_ROC_AUC_Mean,Perm_Test_ROC_AUC_Std,Perm_Train_Recall_Mean,Combined_Rank_Score,Rank
0,hemo,0.317315,0.228928,3.511111e-02,9.999067e-03,0.050286,0.273121,1
1,pcv,0.161478,0.184019,-2.370370e-04,1.185185e-04,0.000000,0.172748,2
2,al,0.121809,0.091813,9.362963e-03,3.760046e-03,0.009143,0.106811,3
3,sg,0.116175,0.070314,2.874074e-03,1.214815e-03,0.013714,0.093245,4
4,dm,0.081528,0.051016,-8.888889e-05,1.357800e-04,0.000000,0.066272,5
5,htn,0.053732,0.076331,-1.777778e-04,1.451549e-04,0.002286,0.065031,6
6,sc,0.030732,0.081393,9.748148e-03,3.767626e-03,0.005714,0.056062,7
7,rc,0.038233,0.069351,2.962963e-04,4.440893e-17,0.002286,0.053792,8
8,bgr,0.025924,0.027921,-3.330669e-17,3.245763e-04,0.003429,0.026922,9
9,wc,0.019390,0.008896,0.000000e+00,0.000000e+00,0.000000,0.014143,10


### 4.5.2. Thực nghiệm định lượng sai lệch do Data Leakage ở các bài làm trên Kaggle
So sánh trực tiếp quy trình chuẩn của nhóm (`make_preprocessor()` fit bên trong từng fold CV và chỉ trên tập Train 280 mẫu) với các lỗi phổ biến trên Kaggle:
1. Điền khuyết bằng trung bình cộng toàn cục (`df.fillna(df.mean())`) làm lệch tâm điền khuyết của Creatinine huyết thanh (`sc`) tới **+136.34%** (`3.0725` so với `1.30 mg/dL`).
2. Chuẩn hóa `StandardScaler` trên toàn bộ 400 mẫu trước khi chia tập làm rò rỉ điểm ngoại lai Kali máu (`pot`) từ tập Test vào tập Train, làm biến dạng độ lệch chuẩn **+324.13%** (`2.8178` so với `0.6644 mEq/L`).
3. Xóa dòng khuyết thiếu (`df.dropna()`) làm mất **60.5% tổng số mẫu (242/400)** và bỏ rơi **82.7% bệnh nhân CKD (62/75 ca)** trên tập kiểm thử 120 mẫu.

In [7]:
df_shift = pd.read_csv(os.path.join(PROJECT_ROOT, 'reports', 'tables', 'ch4_kaggle_imputation_shift.csv'))
df_audit = pd.read_csv(os.path.join(PROJECT_ROOT, 'reports', 'tables', 'ch4_kaggle_leakage_audit.csv'))

print("=== ĐỘ LỆCH THAM SỐ ĐIỀN KHUYẾT VÀ CHUẨN HÓA (KAGGLE LEAKAGE VS STRICT PIPELINE) ===")
display(df_shift)

print("\n=== ĐỐI SÁNH HIỆU NĂNG VÀ ĐỘ BAO PHỦ BỆNH NHÂN ===")
display(df_audit)

img_kag = Image.open(os.path.join(PROJECT_ROOT, 'reports', 'figures', 'ch4_kaggle_leakage_audit.png'))
plt.figure(figsize=(13.5, 5.2))
plt.imshow(img_kag)
plt.axis('off')
plt.tight_layout()
plt.show()

=== ĐỘ LỆCH THAM SỐ ĐIỀN KHUYẾT VÀ CHUẨN HÓA (KAGGLE LEAKAGE VS STRICT PIPELINE) ===


,Feature,Train_Median_Strict,Test_Median_True,Full400_Median_Leaky,Full400_Mean_Leaky_Kaggle,Train_Std_Strict,Full400_Std_Leaky,Imputation_Shift_Pct_MeanVsMed,Scale_Shift_Pct_FullVsTrain
0,sc,1.3,1.2,1.30,3.0725,6.1886,5.6218,136.34,9.16
1,bu,44.0,39.0,42.00,57.4257,48.4332,49.3335,30.51,1.86
2,bgr,120.0,124.5,121.00,148.0365,71.1172,75.1666,23.36,5.69
3,pot,4.3,4.5,4.40,4.6272,0.6644,2.8178,7.61,324.13
4,wc,8000.0,8000.0,8000.00,8406.1224,2710.9368,2526.4298,5.08,6.81
5,age,54.0,55.0,55.00,51.4834,16.2618,16.9618,4.66,4.30
6,bp,80.0,80.0,80.00,76.4691,14.2721,13.4729,4.41,5.60
7,pcv,40.0,40.0,40.00,38.8845,8.2564,8.1520,2.79,1.26
8,rc,4.8,4.7,4.80,4.7074,0.8530,0.8404,1.93,1.48
9,hemo,12.6,12.7,12.65,12.5264,2.7677,2.7131,0.58,1.97



=== ĐỐI SÁNH HIỆU NĂNG VÀ ĐỘ BAO PHỦ BỆNH NHÂN ===


,Algorithm,Strict_CV_Recall,Leaky_PreFit_CV_Recall,Strict_CV_Accuracy,Leaky_PreFit_CV_Accuracy,Test_Recall_All120,Test_Recall_MissingRows,Complete_Cases_Retained_Pct
0,K-Nearest Neighbors (k=5),0.897143,0.902857,0.935714,0.939286,0.893333,0.903226,39.5
1,Decision Tree (CART),0.942857,0.942857,0.953571,0.953571,0.973333,0.967742,39.5
2,Random Forest,0.982857,0.982857,0.978571,0.982143,0.986667,0.983871,39.5
3,XGBoost,0.977143,0.971429,0.975000,0.971429,0.986667,0.983871,39.5
4,Logistic Regression,0.988571,0.988571,0.989286,0.992857,0.986667,0.983871,39.5
5,Support Vector Machine,0.977143,0.982857,0.985714,0.989286,0.973333,0.967742,39.5
